# Session 2: profit = (p - v) q - F
**A. Real-world problem.** A made-up neighbourhood bakery sells loaves. Each month it pays
rent, an oven lease and wages whether or not anyone buys bread. Each loaf also costs flour
and energy. The owner asks four things: how many loaves to sell to break even, how far
sales can fall before the bakery loses money, how many loaves earn a target profit, and
how risky the whole thing is. This session turns the lemonade stall into equations.

*General education only: the bakery and its numbers are made up, and this is not
financial advice.*

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

F = 1200.0        # fixed cost per month, dollars (rent, oven lease, wages)
P = 3.50          # price of one loaf, dollars per loaf
V = 1.40          # variable cost of one loaf, dollars per loaf
Q = 800.0         # expected loaves sold per month
DEMAND_A = 1675.0   # demand estimate: loaves per month at price 0 ...
DEMAND_B = 250.0    # ... minus 250 loaves for every extra dollar of price

## Step 1: profit and contribution margin
Profit = income - cost = p q - (F + v q) = (p - v) q - F.
The **contribution margin** p - v is what one loaf contributes towards the fixed cost.

In [ ]:
def profit(q, p=P, v=V, fixed=F):
    """Monthly profit in dollars for q loaves."""
    return (p - v) * np.asarray(q, dtype=float) - fixed


def contribution_margin(p=P, v=V):
    return p - v


def contribution_ratio(p=P, v=V):
    """Share of each sales dollar left after variable cost."""
    return (p - v) / p

## Step 2: break-even quantity and revenue
Set profit = 0: q* = F / (p - v). The revenue at that point is p q* = F / (1 - v/p).
There is no break-even when p <= v. We check the formula by *scanning* q on a fine grid.

In [ ]:
def break_even_qty(fixed=F, p=P, v=V):
    cm = p - v
    if cm <= 0:
        return None
    return fixed / cm


def break_even_revenue(fixed=F, p=P, v=V):
    q = break_even_qty(fixed, p, v)
    return None if q is None else p * q


def scan_break_even(fixed=F, p=P, v=V, q_max=5000.0, step=0.01):
    """First q on a grid where profit >= 0 (numerical check of the formula)."""
    q = np.arange(0.0, q_max + step, step)
    ok = np.nonzero(profit(q, p, v, fixed) >= -1e-9)[0]
    return float(q[ok[0]]) if len(ok) else None


def whole_units(x):
    """Round up to whole loaves (with a tiny tolerance for round-off)."""
    return int(np.ceil(x - 1e-9))

## Step 3: margin of safety and the target profit
Margin of safety = how far sales can fall before profit is zero, as a share of sales:
(q - q*) / q. For a target profit T: (p - v) q - F = T, so q = (F + T) / (p - v).

In [ ]:
def margin_of_safety(q=Q, fixed=F, p=P, v=V):
    be = break_even_qty(fixed, p, v)
    if be is None or q <= 0:
        return None
    return (q - be) / q


def target_qty(target, fixed=F, p=P, v=V):
    cm = p - v
    if cm <= 0:
        return None
    return (fixed + target) / cm

## Step 4: degree of operating leverage
Fixed costs act like a lever: when sales rise a little, profit rises a lot, and when sales
fall, profit falls a lot. DOL = (contribution margin * q) / profit = (percent change in
profit) / (percent change in sales). It is not defined when profit is zero.

In [ ]:
def dol(q=Q, fixed=F, p=P, v=V):
    pr = float(profit(q, p, v, fixed))
    if abs(pr) < 1e-9:
        return None
    return (p - v) * q / pr


def dol_numeric(q=Q, fixed=F, p=P, v=V, change=0.10):
    """Percent change in profit divided by the percent change in sales, by direct recomputation."""
    p0 = float(profit(q, p, v, fixed))
    p1 = float(profit(q * (1 + change), p, v, fixed))
    return ((p1 - p0) / p0) / change

## Step 5: a design problem - choose the price
Suppose the owner estimates that demand falls as the price rises:
loaves per month = 1675 - 250 * price. Then both the margin and the quantity depend on
the price. We want a price that gives a target profit.

In [ ]:
def demand(price):
    return np.maximum(DEMAND_A - DEMAND_B * np.asarray(price, dtype=float), 0.0)


def profit_at_price(price, fixed=F, v=V):
    return (np.asarray(price, dtype=float) - v) * demand(price) - fixed


def best_price(v=V):
    """Price with the largest profit: zero slope of (p - v)(a - b p)."""
    return (DEMAND_A / DEMAND_B + v) / 2.0


def prices_for_target(target, fixed=F, v=V):
    """Both prices that give exactly `target` profit (empty list if the target cannot be reached)."""
    b1 = (DEMAND_A + DEMAND_B * v) / DEMAND_B
    c1 = (DEMAND_A * v + fixed + target) / DEMAND_B
    disc = b1 * b1 - 4 * c1
    if disc < 0:
        return []
    r = np.sqrt(disc)
    return [(b1 - r) / 2, (b1 + r) / 2]


def scan_prices_for_target(target, fixed=F, v=V, lo=1.5, hi=6.5, step=0.0005):
    """Numerical check: prices where profit crosses the target (linear interpolation on a grid)."""
    pr = np.arange(lo, hi, step)
    d = profit_at_price(pr, fixed, v) - target
    idx = np.nonzero(np.sign(d[:-1]) * np.sign(d[1:]) <= 0)[0]
    return [float(pr[i] - d[i] * step / (d[i + 1] - d[i])) if d[i + 1] != d[i] else float(pr[i]) for i in idx]

## See it
Left: income and cost lines against loaves, with the break-even point and the margin of
safety. Right: profit against price for the design problem.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    q_star = break_even_qty()
    print(f"contribution margin {contribution_margin():.2f} $/loaf, ratio {contribution_ratio():.2f}")
    print(f"break-even {q_star:.2f} loaves (scan {scan_break_even():.2f}), whole loaves {whole_units(q_star)}")
    print(f"break-even revenue ${break_even_revenue():.2f}")
    print(f"profit at {Q:.0f} loaves: ${float(profit(Q)):.2f}, margin of safety {100 * margin_of_safety():.2f} %")
    print(f"loaves for $600 profit: {target_qty(600):.2f} (whole: {whole_units(target_qty(600))})")
    print(f"DOL {dol():.2f}, numeric {dol_numeric():.2f}")
    for ch in (0.10, -0.10):
        q2 = Q * (1 + ch)
        print(f"sales {100 * ch:+.0f} % -> {q2:.0f} loaves, profit ${float(profit(q2)):.2f}")
    t = prices_for_target(500)
    print("prices for $500 profit:", np.round(t, 4), " scan:", np.round(scan_prices_for_target(500), 3))
    print("loaves at those prices:", np.round(demand(t), 1))
    print(f"best price {best_price():.2f}, best profit ${float(profit_at_price(best_price())):.2f}")

    q = np.linspace(0, 1500, 301)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(q, P * q, color="#2a9d8f", label="income")
    ax.plot(q, F + V * q, color="#d9534f", label="total cost")
    ax.plot([q_star], [P * q_star], "ko")
    ax.annotate(f"break-even {q_star:.0f} loaves", (q_star, P * q_star), (150, 3500))
    ax.annotate("", xy=(Q, 300), xytext=(q_star, 300), arrowprops={"arrowstyle": "<->"})
    ax.text(q_star + 10, 380, "margin of safety", fontsize=8)
    ax.set_xlabel("loaves sold per month (loaves)")
    ax.set_ylabel("dollars per month ($)")
    ax.set_title("Income, cost and the break-even point")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left")
    fig.tight_layout()
    fig.savefig(OUT / "cvp_lines.png", dpi=120)

    price = np.linspace(1.5, 6.5, 400)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(price, profit_at_price(price), color="#1f6f8b")
    ax2.axhline(500, color="gray", ls="--", label="target $500")
    ax2.plot(t, [500, 500], "ko")
    ax2.set_xlabel("price (dollars per loaf)")
    ax2.set_ylabel("monthly profit ($)")
    ax2.set_title("Two prices reach the $500 target")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "price_design.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Use `break_even_qty` and `scan_break_even` for p = 4.00 and v = 1.60, with F = 1200. Predict first, then check that the two agree.
2. **Stretch.** Write `sales_needed_for_margin(m)` that finds the sales volume giving a margin of safety of 40 percent, and test it with the worked example's numbers.
3. **Explain.** Explain why two businesses with the same profit can have different degrees of operating leverage, and which one you would prefer in a year with uncertain sales.

## Assessment
- **A. Conceptual.** Two bakeries both earn 480 dollars at 800 loaves. Bakery X is the one in the worked example. Bakery Y has fixed cost 600 dollars and a variable cost of 2.15 dollars per loaf (more hired help, less rent), both at price 3.50. Which is riskier if sales fall by 10 percent, and why?
- **B. Numerical.** A cafe has F = 900 dollars, price 5 dollars and variable cost 2 dollars. Find the break-even quantity and revenue, and the quantity needed for a 600 dollar profit. Check the units.
- **C. Graph and simulation interpretation.** On the profit-against-loaves line of the worked example, what are the value at 0 loaves, the slope, and the point where it crosses zero? What does each mean?
- **D. Design and optimisation challenge.** Design: with demand q = 1675 - 250 p, variable cost 1.40 dollars and fixed cost 1200 dollars, choose a price that gives 500 dollars profit. How many prices work, what do they sell, and can the bakery reach 600 dollars?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    q = L.break_even_qty()
    checks.append(("worked example: contribution margin 3.50 - 1.40 = 2.10 and ratio 0.60",
                   abs(L.contribution_margin() - 2.1) < 1e-12 and abs(L.contribution_ratio() - 0.6) < 1e-12))
    checks.append(("worked example: break-even quantity 1200 / 2.10 = 571.43 loaves, 572 whole loaves",
                   abs(q - 571.4286) < 1e-3 and L.whole_units(q) == 572))
    checks.append(("scanning q on a fine grid finds the same break-even", abs(L.scan_break_even() - q) < 0.011))
    checks.append(("break-even revenue is $2000 = 1200 / 0.60", abs(L.break_even_revenue() - 2000.0) < 1e-9))
    checks.append(("profit is zero at break-even and $480 at 800 loaves",
                   abs(float(L.profit(q))) < 1e-9 and abs(float(L.profit(800)) - 480.0) < 1e-9))
    checks.append(("worked example: margin of safety at 800 loaves is 28.57 percent",
                   abs(L.margin_of_safety() - 0.285714) < 1e-5))
    checks.append(("target profit $600 needs 857.14 loaves, 858 whole loaves",
                   abs(L.target_qty(600) - 857.142857) < 1e-5 and L.whole_units(L.target_qty(600)) == 858
                   and float(L.profit(858)) >= 600 > float(L.profit(857))))
    checks.append(("worked example: degree of operating leverage is 3.5 (formula and recomputation)",
                   abs(L.dol() - 3.5) < 1e-9 and abs(L.dol_numeric() - 3.5) < 1e-9))
    checks.append(("sales +10 percent gives profit $648 (+35 percent)",
                   abs(float(L.profit(880)) - 648.0) < 1e-9 and abs(float(L.profit(880)) / 480.0 - 1.35) < 1e-9))
    checks.append(("sales -10 percent gives profit $312 (-35 percent)",
                   abs(float(L.profit(720)) - 312.0) < 1e-9 and abs(float(L.profit(720)) / 480.0 - 0.65) < 1e-9))
    # hand calculation: F=900, p=5, v=2 -> cm 3
    checks.append(("hand calculation: F=900, p=5, v=2 gives 300 loaves, revenue $1500, 500 loaves for $600",
                   abs(L.break_even_qty(900, 5, 2) - 300) < 1e-9 and abs(L.break_even_revenue(900, 5, 2) - 1500) < 1e-9
                   and abs(L.target_qty(600, 900, 5, 2) - 500) < 1e-9))
    checks.append(("profit line: -$1200 at zero loaves, slope $2.10 per loaf",
                   abs(float(L.profit(0)) + 1200) < 1e-9 and abs(float(L.profit(101) - L.profit(100)) - 2.1) < 1e-9))
    # risk comparison with the same profit at 800 loaves
    q2 = L.profit(720, 3.5, 2.15, 600.0)
    checks.append(("a low-fixed-cost bakery with the same $480 profit loses only 22.5 percent when sales fall 10 percent",
                   abs(float(L.profit(800, 3.5, 2.15, 600.0)) - 480.0) < 1e-9 and abs(float(q2) - 372.0) < 1e-9
                   and abs(float(q2) / 480.0 - 0.775) < 1e-9 and abs(L.dol(800, 600.0, 3.5, 2.15) - 2.25) < 1e-9))
    # design problem
    t = L.prices_for_target(500)
    checks.append(("design: two prices, 3.58 and 4.52 dollars, reach a $500 profit",
                   len(t) == 2 and abs(t[0] - 3.5783) < 1e-3 and abs(t[1] - 4.5217) < 1e-3
                   and all(abs(float(L.profit_at_price(x)) - 500) < 1e-9 for x in t)))
    s = L.scan_prices_for_target(500)
    checks.append(("design: scanning prices finds the same two roots", len(s) == 2 and all(abs(a - b) < 2e-3 for a, b in zip(s, t))))
    d = L.demand(t)
    checks.append(("design: the two prices sell 780 and 545 loaves", abs(d[0] - 780.4) < 0.1 and abs(d[1] - 544.6) < 0.1))
    mos = [L.margin_of_safety(float(d[i]), 1200, t[i], 1.4) for i in range(2)]
    be = [L.break_even_qty(1200, t[i], 1.4) for i in range(2)]
    checks.append(("design: both prices have break-even near 551 and 384 loaves and margin of safety 500/1700 = 29.4 percent",
                   abs(be[0] - 550.9) < 0.1 and abs(be[1] - 384.4) < 0.1
                   and all(abs(m - 500 / 1700) < 1e-9 for m in mos)))
    checks.append(("design: best price is $4.05 with profit about $555.6",
                   abs(L.best_price() - 4.05) < 1e-12 and abs(float(L.profit_at_price(4.05)) - 555.625) < 1e-9))
    grid = np.arange(1.5, 6.5, 0.01)
    checks.append(("design: a scan over prices agrees the best price is 4.05",
                   abs(float(grid[np.argmax(L.profit_at_price(grid))]) - 4.05) < 0.011))
    checks.append(("design: a $600 target cannot be reached at any price", L.prices_for_target(600) == []
                   and float(np.max(L.profit_at_price(grid))) < 600))
    # edge and limit cases
    checks.append(("limit: price equal to variable cost has no break-even", L.break_even_qty(1200, 1.4, 1.4) is None
                   and L.break_even_revenue(1200, 1.0, 1.4) is None and L.scan_break_even(1200, 1.4, 1.4) is None))
    checks.append(("limit: DOL is not defined when profit is exactly zero", L.dol(q) is None))
    checks.append(("limit: margin of safety is negative below break-even", L.margin_of_safety(400) < 0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
